In [6]:
import os
from google.cloud import bigquery
import pandas as pd
import requests
from datetime import datetime
from datetime import timezone


In [7]:
def load_all_stations_historical_to_bigquery(project_id, dataset_id, table_id, force=False):
    client = bigquery.Client(project=project_id)
    table_full_path = f"{project_id}.{dataset_id}.{table_id}"

    # --- Vérifie si la table contient déjà des données ---
    if not force:
        try:
            table = client.get_table(table_full_path)
            if table.num_rows > 0:
                print(
                    f" Historique déjà chargé ({table.num_rows} lignes) — étape ignorée. "
                    f"Utilise force=True pour forcer le rechargement."
                )
                return None
        except Exception:
            # La table n'existe pas encore, on continue normalement
            pass

    url = (
        "https://portail-api-data.montpellier.fr/ngsi-ld/v1/temporal/entities"
        "?type=BikeHireDockingStation"
        "&format=temporalValues"
        "&timerel=after"
        "&timeAt=2025-12-31T23%3A59%3A59Z"
    )

    response = requests.get(url, timeout=15)

    if response.status_code in [200, 206]:
        entities = response.json()
        data_clean = []

        for entity in entities:
            full_id = entity.get("id", "")
            station_id = full_id.split(":")[-1] if ":" in full_id else full_id
            station_id = str(station_id).zfill(3)  # <-- cohérence avec stations_referentiel

            available_bike = entity.get("availableBikeNumber", {})
            values = available_bike.get("values", [])

            for bike_number, date in values:
                data_clean.append(
                    {
                        "station_id": station_id,
                        "availableBikeNumber": int(bike_number)
                        if bike_number is not None
                        else None,
                        "date": date,
                    }
                )

        df = pd.DataFrame(data_clean)
        df["date"] = pd.to_datetime(df["date"])

        job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")

        job = client.load_table_from_dataframe(
            df, table_full_path, job_config=job_config
        )
        job.result()

        print(
            f" Données insérées dans BigQuery ! ({len(df)} lignes, table: {table_id})"
        )
        return df
    else:
        print(f" Erreur HTTP {response.status_code} : {response.text}")
        return None

In [8]:
def load_latest_stations_to_bigquery(project_id, dataset_id, table_id):
    url = "https://portail-api-data.montpellier.fr/ngsi-ld/v1/entities?type=BikeHireDockingStation"

    response = requests.get(url)

    if response.status_code == 200:
        entities = response.json()
        data_clean = []
        now = datetime.now(timezone.utc)

        for entity in entities:
            full_id = entity.get("id", "")
            station_id = full_id.split(":")[-1] if ":" in full_id else full_id

            available_bike = (
                entity.get("availableBikeNumber", {}).get("value", 0)
            )
            free_slots = entity.get("freeSlotNumber", {}).get("value", 0)
            status = entity.get("status", {}).get("value", "Working")

            # Récupération de la date de l'API ou date actuelle UTC
            date_obs = entity.get("observationDateTime", {}).get("value", None)
            if not date_obs:
                date_obs = now

            data_clean.append(
                {
                    "station_id": str(station_id),
                    "availableBikeNumber": int(available_bike)
                    if available_bike is not None
                    else 0,
                    "freeSlotNumber": int(free_slots)
                    if free_slots is not None
                    else 0,
                    "status": str(status),
                    "date": date_obs,
                }
            )

        df_latest = pd.DataFrame(data_clean)
        df_latest["date"] = pd.to_datetime(df_latest["date"])

        # --- ENVOI DIRECT DANS BIGQUERY ---
        client = bigquery.Client(project=project_id)
        table_full_path = f"{project_id}.{dataset_id}.{table_id}"

        # Configuration : 'WRITE_APPEND' ajoute les lignes sans écraser la table
        job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND",  # important : ajoute les lignes sans écraser la table
        schema_update_options=[
        bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION
        ],
        )

        # Envoi du DataFrame vers BigQuery
        job = client.load_table_from_dataframe(
            df_latest, table_full_path, job_config=job_config
        )
        job.result()  # Bloque jusqu'à la fin de l'injection

        print(
            f" Temps réel injecté dans BigQuery : {len(df_latest)} stations."
        )
        return df_latest
    else:
        print(f" Erreur HTTP {response.status_code}")
        return None

In [9]:
def load_stations_to_bigquery(project_id, dataset_id):
    url_stations = "https://gbfs.theta.fifteen.eu/gbfs/2.2/montpellier/en/station_information.json"
    response = requests.get(url_stations).json()

    # Extraction des données
    stations_list = response["data"]["stations"]
    df_locations = pd.DataFrame(stations_list)

    # Conservation des colonnes clés
    df_locations = df_locations[["station_id", "name", "lat", "lon"]]

    # Conversion du station_id en format propre avec zfill
    df_locations["station_id"] = (
        df_locations["station_id"].astype(str).str.zfill(3)
    )

    # --- ENVOI DIRECT DANS BIGQUERY (SANS PANDAS-GBQ) ---
    client = bigquery.Client(project=project_id)
    table_full_path = f"{project_id}.{dataset_id}.stations_referentiel"

    # 'WRITE_TRUNCATE' permet d'écraser la table du référentiel pour la garder toujours à jour
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

    job = client.load_table_from_dataframe(
        df_locations, table_full_path, job_config=job_config
    )
    job.result()  # Attend la fin du traitement

    print(
        f" Référentiel des stations mis à jour dans BigQuery ! ({len(df_locations)} stations)"
    )
    return df_locations

In [10]:
PROJECT_ID = "prediction-velo"  # L'ID de ton projet GCP
DATASET_ID = "prediction_velo_raw"
TABLE_ID = "realtime"

# 1. Tu lances d'abord l'historique une seule fois pour charger le passé
load_all_stations_historical_to_bigquery(PROJECT_ID, DATASET_ID, TABLE_ID)

# 2. Tu peux tester l'ajout d'un relevé instantané
load_latest_stations_to_bigquery(PROJECT_ID, DATASET_ID, TABLE_ID)

load_stations_to_bigquery(PROJECT_ID, DATASET_ID)

 Historique déjà chargé (2600104 lignes) — étape ignorée. Utilise force=True pour forcer le rechargement.


C:\Users\foute\Desktop\PredictionVelo\prediction-velo-tam\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


 Temps réel injecté dans BigQuery : 52 stations.


C:\Users\foute\Desktop\PredictionVelo\prediction-velo-tam\.venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


 Référentiel des stations mis à jour dans BigQuery ! (52 stations)


,station_id,name,lat,lon
0,001,Rue Jules Ferry - Gare Saint-Roch,43.605366,3.881346
1,002,Comédie,43.608148,3.878778
2,004,Hôtel de Ville,43.599088,3.894866
3,005,Corum,43.613989,3.881600
4,006,Place Albert 1er - St Charles,43.616768,3.873375
5,007,Foch,43.610989,3.873345
6,009,Observatoire,43.606301,3.877240
7,010,Rondelet,43.603038,3.875796
8,011,Plan Cabanes,43.608491,3.868389
9,012,Boutonnet,43.622629,3.868375
